# Step 08 - Apache Iceberg Features

## Mục tiêu

Trong bước này, hệ thống sẽ kiểm chứng và thực hành các tính năng chính của Apache Iceberg, bao gồm:

- Theo dõi lịch sử thay đổi của bảng bằng Snapshot và History.
- Thực hiện các thao tác INSERT, UPDATE và DELETE trên Iceberg table.
- Sử dụng MERGE INTO để hỗ trợ cập nhật dữ liệu theo kiểu incremental.
- Sử dụng Time Travel để truy vấn lại dữ liệu tại các snapshot trước đó.
- Thực hiện Schema Evolution để thay đổi cấu trúc bảng mà không làm mất dữ liệu hiện có.
- Khám phá các metadata table của Iceberg như snapshots, history, files và manifests.
- Hiểu cách Iceberg quản lý phiên bản dữ liệu và metadata trên MinIO.

In [3]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:{ICEBERG_VERSION},"
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
    f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}"
)

AWS_ACCESS_KEY_ID = (
    os.environ.get("AWS_ACCESS_KEY_ID")
    or os.environ.get("MINIO_ROOT_USER")
)

AWS_SECRET_ACCESS_KEY = (
    os.environ.get("AWS_SECRET_ACCESS_KEY")
    or os.environ.get("MINIO_ROOT_PASSWORD")
)

spark = (
    SparkSession.builder
    .appName("EcommerceIcebergFeatures")
    .master("local[*]")

    .config("spark.jars.packages", PACKAGES)

    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.lakehouse.type",
        "rest"
    )
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )

    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        AWS_ACCESS_KEY_ID
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        AWS_SECRET_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .getOrCreate()
)

print("Phiên bản Spark:", spark.version)

Phiên bản Spark: 4.1.3


In [6]:
spark.sql("""
DROP TABLE IF EXISTS lakehouse.gold.iceberg_demo
""")

DataFrame[]

In [7]:
spark.sql("""
CREATE TABLE lakehouse.gold.iceberg_demo (
    id INT,
    product_name STRING,
    price DOUBLE,
    status STRING
)
USING iceberg
""")

DataFrame[]

In [8]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
""").show()

+---+------------+-----+------+
| id|product_name|price|status|
+---+------------+-----+------+
+---+------------+-----+------+



In [9]:
spark.sql("""
INSERT INTO lakehouse.gold.iceberg_demo
VALUES
    (1, 'Laptop', 1200.0, 'active'),
    (2, 'Mouse', 25.0, 'active'),
    (3, 'Keyboard', 50.0, 'active')
""")

DataFrame[]

In [12]:
spark.sql(
    "select * from lakehouse.gold.iceberg_demo"
).show()

+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  25.0|active|
|  3|    Keyboard|  50.0|active|
+---+------------+------+------+



In [15]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""").show(
    truncate=False
)

+----------------------+-------------------+---------+
|committed_at          |snapshot_id        |operation|
+----------------------+-------------------+---------+
|2026-10-04 11:53:51.96|4837729842680469689|append   |
+----------------------+-------------------+---------+



In [17]:
spark.sql("""
select *
from lakehouse.gold.iceberg_demo.history
order by made_current_at
""").show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-10-04 11:53:...|4837729842680469689|     NULL|               true|
+--------------------+-------------------+---------+-------------------+



In [18]:
spark.sql("""
UPDATE lakehouse.gold.iceberg_demo
SET price = 30.0
WHERE id = 2
""")

DataFrame[]

In [19]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
ORDER BY id
""").show()

+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  30.0|active|
|  3|    Keyboard|  50.0|active|
+---+------------+------+------+



In [20]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""").show(
    truncate=False
)

+----------------------+-------------------+---------+
|committed_at          |snapshot_id        |operation|
+----------------------+-------------------+---------+
|2026-10-04 11:53:51.96|4837729842680469689|append   |
|2026-10-04 12:08:29.54|4551194912360874147|overwrite|
+----------------------+-------------------+---------+



In [21]:
spark.sql("""
DELETE FROM lakehouse.gold.iceberg_demo
WHERE id = 3
""")

DataFrame[]

In [22]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
ORDER BY id
""").show()

+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  30.0|active|
+---+------------+------+------+



In [23]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""").show(
    truncate=False
)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-04 11:53:51.96 |4837729842680469689|append   |
|2026-10-04 12:08:29.54 |4551194912360874147|overwrite|
|2026-10-04 12:09:28.979|8818790664808459275|delete   |
+-----------------------+-------------------+---------+



In [24]:
updates = spark.createDataFrame(
    [
        (2, "Mouse", 35.0, "active"),
        (4, "Monitor", 250.0, "active")
    ],
    [
        "id",
        "product_name",
        "price",
        "status"
    ]
)

In [25]:
updates.createOrReplaceTempView(
    "product_updates"
)

In [26]:
spark.sql("""
MERGE INTO lakehouse.gold.iceberg_demo AS target

USING product_updates AS source

ON target.id = source.id

WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.price = source.price,
    target.status = source.status

WHEN NOT MATCHED THEN INSERT (
    id,
    product_name,
    price,
    status
)
VALUES (
    source.id,
    source.product_name,
    source.price,
    source.status
)
""")

DataFrame[]

In [27]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
ORDER BY id
""").show()

+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  35.0|active|
|  4|     Monitor| 250.0|active|
+---+------------+------+------+



In [28]:
snapshots = spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""")

snapshots.show(
    truncate=False
)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-04 11:53:51.96 |4837729842680469689|append   |
|2026-10-04 12:08:29.54 |4551194912360874147|overwrite|
|2026-10-04 12:09:28.979|8818790664808459275|delete   |
|2026-10-04 12:54:09.362|6264797594447724430|overwrite|
+-----------------------+-------------------+---------+



In [29]:
snapshots = spark.sql("""
SELECT
    snapshot_id,
    committed_at
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""")

snapshots.show(
    truncate=False
)

+-------------------+-----------------------+
|snapshot_id        |committed_at           |
+-------------------+-----------------------+
|4837729842680469689|2026-10-04 11:53:51.96 |
|4551194912360874147|2026-10-04 12:08:29.54 |
|8818790664808459275|2026-10-04 12:09:28.979|
|6264797594447724430|2026-10-04 12:54:09.362|
+-------------------+-----------------------+



In [30]:
snapshot_rows = snapshots.collect()

first_snapshot_id = (
    snapshot_rows[0]["snapshot_id"]
)

print(
    "First snapshot:",
    first_snapshot_id
)

First snapshot: 4837729842680469689


In [32]:
first_snapshot_id = snapshot_rows[0]["snapshot_id"]

print("First snapshot:", first_snapshot_id)

old_data = spark.sql(f"""
SELECT *
FROM lakehouse.gold.iceberg_demo
VERSION AS OF {first_snapshot_id}
""")

old_data.orderBy("id").show()

First snapshot: 4837729842680469689
+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  25.0|active|
|  3|    Keyboard|  50.0|active|
+---+------------+------+------+



In [33]:
print("===== CURRENT =====")

spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
ORDER BY id
""").show()

===== CURRENT =====
+---+------------+------+------+
| id|product_name| price|status|
+---+------------+------+------+
|  1|      Laptop|1200.0|active|
|  2|       Mouse|  35.0|active|
|  4|     Monitor| 250.0|active|
+---+------------+------+------+



In [35]:
spark.sql("""
ALTER TABLE lakehouse.gold.iceberg_demo
ADD COLUMN category STRING
""")

DataFrame[]

In [36]:
spark.table(
    "lakehouse.gold.iceberg_demo"
).printSchema()

root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- category: string (nullable = true)



In [37]:
spark.sql("""
select *
from lakehouse.gold.iceberg_demo
""").show()

+---+------------+------+------+--------+
| id|product_name| price|status|category|
+---+------------+------+------+--------+
|  1|      Laptop|1200.0|active|    NULL|
|  2|       Mouse|  35.0|active|    NULL|
|  4|     Monitor| 250.0|active|    NULL|
+---+------------+------+------+--------+



In [38]:
spark.sql("""
UPDATE lakehouse.gold.iceberg_demo
SET category = 'electronics'
WHERE id IN (1, 4)
""")

DataFrame[]

In [39]:
spark.sql("""
select *
from lakehouse.gold.iceberg_demo
""").show()

+---+------------+------+------+-----------+
| id|product_name| price|status|   category|
+---+------------+------+------+-----------+
|  1|      Laptop|1200.0|active|electronics|
|  2|       Mouse|  35.0|active|       NULL|
|  4|     Monitor| 250.0|active|electronics|
+---+------------+------+------+-----------+



In [41]:
spark.sql("""
SELECT
    content,
    file_path,
    file_format,
    record_count,
    file_size_in_bytes
FROM lakehouse.gold.iceberg_demo.files
""").show(
    truncate=False, vertical=True
)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------
 content            | 0                                                                                                                       
 file_path          | s3://ecommerce-lakehouse/warehouse/gold/iceberg_demo/data/00000-57-f2fd1e64-5a2c-4512-ae56-907e3c4bfee2-0-00001.parquet 
 file_format        | PARQUET                                                                                                                 
 record_count       | 3                                                                                                                       
 file_size_in_bytes | 1633                                                                                                                    



In [43]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo.manifests
""").show(
    truncate=False, vertical=True
)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------
 content                     | 0                                                                                                          
 path                        | s3://ecommerce-lakehouse/warehouse/gold/iceberg_demo/metadata/edb897be-a614-44f1-9893-f6cbcb4d7556-m2.avro 
 length                      | 7289                                                                                                       
 partition_spec_id           | 0                                                                                                          
 added_snapshot_id           | 8054700964926600904                                                                                        
 added_data_files_count      | 1                                                                                                          
 existing_data_files_count 

In [45]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo.metadata_log_entries
ORDER BY timestamp
""").show(
    truncate=False, vertical=True
)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 timestamp              | 2026-10-04 11:53:24.503                                                                                                
 file                   | s3://ecommerce-lakehouse/warehouse/gold/iceberg_demo/metadata/00000-aeaa6070-e113-42e0-b983-315873c5606c.metadata.json 
 latest_snapshot_id     | NULL                                                                                                                   
 latest_schema_id       | NULL                                                                                                                   
 latest_sequence_number | NULL                                                                                                                   
-RECORD 1-------------------------------------------------------------------------------------------------------------------

In [46]:
spark.table(
    "lakehouse.gold.iceberg_demo"
).printSchema()

root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- category: string (nullable = true)



In [47]:
spark.sql("""
SELECT *
FROM lakehouse.gold.iceberg_demo
ORDER BY id
""").show(
    truncate=False
)

+---+------------+------+------+-----------+
|id |product_name|price |status|category   |
+---+------------+------+------+-----------+
|1  |Laptop      |1200.0|active|electronics|
|2  |Mouse       |35.0  |active|NULL       |
|4  |Monitor     |250.0 |active|electronics|
+---+------------+------+------+-----------+



In [48]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.iceberg_demo.snapshots
ORDER BY committed_at
""").show(
    truncate=False
)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-04 11:53:51.96 |4837729842680469689|append   |
|2026-10-04 12:08:29.54 |4551194912360874147|overwrite|
|2026-10-04 12:09:28.979|8818790664808459275|delete   |
|2026-10-04 12:54:09.362|6264797594447724430|overwrite|
|2026-10-04 13:03:07.633|8054700964926600904|overwrite|
+-----------------------+-------------------+---------+



## Kết luận

Trong bước này, hệ thống đã kiểm chứng các tính năng chính của Apache Iceberg:

- Snapshot Management
- Table History
- UPDATE
- DELETE
- MERGE INTO
- Time Travel
- Schema Evolution
- Metadata Tables

Qua đó có thể thấy Iceberg không chỉ lưu dữ liệu mà còn cung cấp khả năng quản lý phiên bản, thay đổi schema và truy vấn lịch sử dữ liệu trên Data Lakehouse.